# Attrition Survival Analysis — Use Case #2

Pendekatan **survival analysis** untuk attrition: bukan sekadar "keluar atau
tidak", tetapi **kapan** karyawan cenderung keluar.

> **Data:** `data/hr_attrition_survival.csv` **sintetis**
> (`scripts/generate_attrition_survival_data.py`).
> `tenure_bulan` = lama observasi, `event_keluar` = 1 bila benar-benar keluar
> (0 = disensor, artinya masih bekerja saat data diambil).

## Mengapa survival analysis?

- Menangani **censoring** (karyawan yang belum keluar tetap memberi informasi).
- Menghasilkan **kurva bertahan** dan **hazard ratio** — lebih kaya daripada
  klasifikasi biner biasa.

Metode: Kaplan–Meier, uji log-rank, dan Cox Proportional Hazards (`lifelines`).

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_attrition_survival.csv", "../data/hr_attrition_survival.csv",
                   "hr_attrition_survival.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_attrition_survival.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
ARTIFACT_DIR = PROJECT_ROOT / "models" / "survival_attrition" / "v1"

DURATION = "tenure_bulan"
EVENT = "event_keluar"
GROUP_COLS = ["departemen", "pendidikan"]

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)
print("Data :", DATA_PATH)
print("Artefak:", ARTIFACT_DIR)

Data : data/hr_attrition_survival.csv
Artefak: /workspace/project/hr-intelligence-system/models/survival_attrition/v1


## 1. Load & Audit

In [2]:
df = pd.read_csv(DATA_PATH)

print("Baris:", len(df))
print(f"Event (keluar)   : {int(df[EVENT].sum())} ({df[EVENT].mean():.1%})")
print(f"Disensor         : {int((1 - df[EVENT]).sum())} ({(1 - df[EVENT]).mean():.1%})")
print(f"Median tenure    : {df[DURATION].median():.0f} bulan")
print(f"Rentang tenure   : {df[DURATION].min()}–{df[DURATION].max()} bulan")
df.head(3)

Baris: 1200
Event (keluar)   : 813 (67.8%)
Disensor         : 387 (32.2%)
Median tenure    : 12 bulan
Rentang tenure   : 1–60 bulan


,employee_id,usia,departemen,pendidikan,gaji_bulanan,jam_lembur_bulanan,kepuasan_kerja,bulan_sejak_promosi,jarak_km,tenure_bulan,event_keluar
0,EMP00001,24,Sales,S1,10955000,10.6,4.09,0.0,8.0,5,1
1,EMP00002,48,Legal,S1,10865000,6.2,3.60,29.7,17.5,25,1
2,EMP00003,44,Sales,S1,8888000,28.3,3.80,0.0,22.9,3,1


## 2. Kaplan–Meier — Kurva Bertahan Keseluruhan

Kurva KM memperkirakan probabilitas karyawan **masih bertahan** setelah t bulan,
dengan memperhitungkan censoring.

In [3]:
kmf = KaplanMeierFitter()
kmf.fit(df[DURATION], event_observed=df[EVENT], label="Semua karyawan")

plt.figure(figsize=(8, 5))
kmf.plot_survival_function(ci_show=True)
plt.xlabel("Tenure (bulan)"); plt.ylabel("Probabilitas masih bertahan")
plt.title("Kurva Kaplan–Meier — keseluruhan")
plt.tight_layout(); plt.show()

for t in [6, 12, 24, 36]:
    print(f"Bertahan >= {t:>2} bulan: {kmf.predict(t):.1%}")
print(f"\nMedian survival: {kmf.median_survival_time_} bulan")

Bertahan >=  6 bulan: 60.2%
Bertahan >= 12 bulan: 49.3%
Bertahan >= 24 bulan: 39.5%
Bertahan >= 36 bulan: 36.2%

Median survival: 12.0 bulan


## 3. Kaplan–Meier per Departemen + Uji Log-Rank

Uji log-rank menguji apakah kurva bertahan antar kelompok **berbeda signifikan**.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, col in zip(axes, GROUP_COLS):
    kmf_g = KaplanMeierFitter()
    for g in sorted(df[col].unique()):
        sub = df[df[col] == g]
        kmf_g.fit(sub[DURATION], event_observed=sub[EVENT], label=f"{g} (n={len(sub)})")
        kmf_g.plot_survival_function(ax=ax, ci_show=False)
    ax.set_xlabel("Tenure (bulan)"); ax.set_ylabel("Probabilitas bertahan")
    ax.set_title(f"Kaplan–Meier per {col}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for col in GROUP_COLS:
    res = multivariate_logrank_test(df[DURATION], df[col], df[EVENT])
    print(f"Log-rank {col:<12}: statistik={res.test_statistic:.2f}, p={res.p_value:.4f} "
          f"→ {'BEDA signifikan' if res.p_value < 0.05 else 'tidak beda signifikan'}")

Log-rank departemen  : statistik=1235.50, p=0.0000 → BEDA signifikan
Log-rank pendidikan  : statistik=1.44, p=0.6970 → tidak beda signifikan


## 4. Cox Proportional Hazards

Cox PH mengestimasi **hazard ratio (HR)** tiap fitur:

- HR > 1 → menaikkan risiko keluar.
- HR < 1 → menurunkan risiko (protektif).

In [5]:
# Siapkan matriks desain (one-hot untuk kategorikal).
cox_df = pd.get_dummies(
    df[[DURATION, EVENT, "usia", "gaji_bulanan", "jam_lembur_bulanan",
        "kepuasan_kerja", "bulan_sejak_promosi", "jarak_km", "departemen", "pendidikan"]],
    columns=["departemen", "pendidikan"], drop_first=True, dtype=float)

cph = CoxPHFitter(penalizer=0.05)
cph.fit(cox_df, duration_col=DURATION, event_col=EVENT)

ringkas_cox = cph.summary[["coef", "exp(coef)", "p"]].copy()
ringkas_cox.columns = ["coef", "hazard_ratio", "p"]
print(ringkas_cox.round(4).to_string())

print(f"\nConcordance index (C-index): {cph.concordance_index_:.4f}")
print("(0,5 = acak, 1,0 = sempurna)")

                         coef  hazard_ratio       p
covariate                                          
usia                   0.0010        1.0010  0.7640
gaji_bulanan          -0.0000        1.0000  0.4565
jam_lembur_bulanan     0.0011        1.0011  0.7701
kepuasan_kerja        -0.0548        0.9466  0.2047
bulan_sejak_promosi    0.0006        1.0006  0.7956
jarak_km               0.0007        1.0007  0.8758
departemen_HR         -0.9046        0.4047  0.0000
departemen_IT          2.2522        9.5091  0.0000
departemen_Legal      -0.9107        0.4022  0.0000
departemen_Marketing   2.0049        7.4253  0.0000
departemen_Operations  1.6681        5.3019  0.0000
departemen_Sales       2.5828       13.2337  0.0000
pendidikan_S1          0.0692        1.0717  0.4099
pendidikan_S2          0.0090        1.0090  0.9381
pendidikan_SMA         0.1759        1.1923  0.0903

Concordance index (C-index): 0.8144
(0,5 = acak, 1,0 = sempurna)


## 5. Visualisasi Hazard Ratio

In [6]:
hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
hr = hr.sort_values("exp(coef)")
hr.columns = ["HR", "HR_low", "HR_high", "p"]

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(hr))
ax.errorbar(hr["HR"], y,
            xerr=[hr["HR"] - hr["HR_low"], hr["HR_high"] - hr["HR"]],
            fmt="o", color="#4C78A8", ecolor="grey", capsize=3)
ax.axvline(1.0, ls="--", color="crimson", label="HR = 1 (netral)")
ax.set_yticks(y); ax.set_yticklabels(hr.index)
ax.set_xlabel("Hazard ratio (95% CI)")
ax.set_title("Cox PH — hazard ratio per fitur")
ax.legend(); plt.tight_layout(); plt.show()

print(hr.round(3).to_string())

                           HR  HR_low  HR_high      p
covariate                                            
departemen_Legal        0.402   0.294    0.550  0.000
departemen_HR           0.405   0.298    0.550  0.000
kepuasan_kerja          0.947   0.870    1.030  0.205
gaji_bulanan            1.000   1.000    1.000  0.456
bulan_sejak_promosi     1.001   0.996    1.006  0.796
jarak_km                1.001   0.992    1.009  0.876
usia                    1.001   0.995    1.007  0.764
jam_lembur_bulanan      1.001   0.994    1.008  0.770
pendidikan_S2           1.009   0.804    1.267  0.938
pendidikan_S1           1.072   0.909    1.264  0.410
pendidikan_SMA          1.192   0.973    1.462  0.090
departemen_Operations   5.302   4.208    6.681  0.000
departemen_Marketing    7.425   5.819    9.474  0.000
departemen_IT           9.509   7.484   12.082  0.000
departemen_Sales       13.234  10.345   16.929  0.000


## 6. Ringkasan Risiko per Departemen

Ambil HR relatif departemen dari model Cox untuk melihat kelompok mana yang
paling berisiko.

In [7]:
dept_hr = hr[hr.index.str.startswith("departemen_")].copy()
dept_hr.index = dept_hr.index.str.replace("departemen_", "")
print("Hazard ratio per departemen (relatif terhadap baseline):")
print(dept_hr.round(3).to_string())

plt.figure(figsize=(7, 4))
sns.barplot(x=dept_hr["HR"], y=dept_hr.index, color="#F58518")
plt.axvline(1.0, ls="--", color="grey")
plt.xlabel("Hazard ratio"); plt.title("Risiko keluar relatif per departemen")
plt.tight_layout(); plt.show()

Hazard ratio per departemen (relatif terhadap baseline):
                HR  HR_low  HR_high    p
covariate                               
Legal        0.402   0.294    0.550  0.0
HR           0.405   0.298    0.550  0.0
Operations   5.302   4.208    6.681  0.0
Marketing    7.425   5.819    9.474  0.0
IT           9.509   7.484   12.082  0.0
Sales       13.234  10.345   16.929  0.0


## 7. Simpan Artifact

In [8]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
import joblib
joblib.dump(cph, ARTIFACT_DIR / "cox_model.pkl")

metadata = {
    "versi_model": "v1",
    "model": "Cox Proportional Hazards (lifelines)",
    "tujuan": "Survival analysis attrition (kapan karyawan keluar)",
    "duration_col": DURATION,
    "event_col": EVENT,
    "n": int(len(df)),
    "n_event": int(df[EVENT].sum()),
    "c_index": round(float(cph.concordance_index_), 4),
    "median_survival_bulan": float(kmf.median_survival_time_),
    "logrank_p": {c: round(float(multivariate_logrank_test(df[DURATION], df[c], df[EVENT]).p_value), 4)
                  for c in GROUP_COLS},
    "hazard_ratio": {k: round(float(v), 4) for k, v in cph.summary["exp(coef)"].items()},
    "dilatih_pada": pd.Timestamp.utcnow().isoformat(),
    "dataset": str(DATA_PATH),
    "dataset_sintetis": True,
}
(ARTIFACT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Tersimpan:", ARTIFACT_DIR)
print(f"C-index: {metadata['c_index']}")

Tersimpan: /workspace/project/hr-intelligence-system/models/survival_attrition/v1
C-index: 0.8144


## 8. Kesimpulan

- Kurva KM menunjukkan probabilitas bertahan menurun seiring tenure; median
  survival tercatat di atas.
- Uji log-rank menunjukkan apakah departemen/pendidikan berhubungan dengan
  waktu keluar.
- Cox PH memberi hazard ratio yang bisa diterjemahkan langsung ke kebijakan
  retensi (mis. memperbaiki kepuasan kerja, mengurangi lembur).

> Ingat: data ini sintetis. Arah hubungan (mis. kepuasan rendah → risiko tinggi)
> masuk akal, tetapi angka pastinya harus divalidasi pada data nyata.